# Campagne promotionnelle FUNPARK : Nettoyage des données

---
# 1. Import des modules et données
---

In [1]:
# Import des modules 
import pandas as pd 
import numpy as np
import os

# Import des données
df_raw = pd.read_excel("../data/raw/funpark_data_raw.xlsx") 

# Copie de travail 
df_clean = df_raw.copy()

---
# 2. Aperçu des données
---

In [2]:
df_clean.head()

,Visiteur,Email,Pays,Type_Billet,Âge,Prix_Billet,Dépenses_Restaurant,Description,Date_Visite,Période
0,tom gonzalez,tom.gonzalez1447@yahoo.fr,France,std,30,NaN,26.87,Churros,2025-01-04,Avant
1,Zoe Gonzalez,zoe.gonzalez1114@outlook.com,Maroc,Etud,25,24.18,4.55,Menu Enfant,2025-04-11,Après
2,Lina Gonzalez,lina.gonzalez1064@gmail.com,Espagne,Standard,48,43.14,11.7,Glace Vanille,2025-03-25,Après
3,Fatima Lefevre,fatima.lefevre2287@mail.com,Allemagne,std,33,49.58,12.33,Café Espresso,2025-02-04,Avant
4,Sami Diaz,sami.diaz1537@gmail.com,Espagne,STD,18,57.25,13.06,Gaufre,2025-06-18,Après


In [3]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2505 entries, 0 to 2504
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Visiteur             2505 non-null   object
 1   Email                2505 non-null   object
 2   Pays                 2505 non-null   object
 3   Type_Billet          2505 non-null   object
 4   Âge                  2362 non-null   object
 5   Prix_Billet          2411 non-null   object
 6   Dépenses_Restaurant  2403 non-null   object
 7   Description          2505 non-null   object
 8   Date_Visite          2505 non-null   object
 9   Période              2505 non-null   object
dtypes: object(10)
memory usage: 195.8+ KB


In [4]:
df_clean.describe()

,Visiteur,Email,Pays,Type_Billet,Âge,Prix_Billet,Dépenses_Restaurant,Description,Date_Visite,Période
count,2505,2505,2505,2505,2362,2411.00,2403,2505,2505,2505
unique,1195,2441,8,17,66,2034.00,1724,15,181,2
top,Yasmine Durand,emma.bennani1186@mail.com,France,Standard,31,43.97,?,Menu Enfant,2025-01-08,Après
freq,10,3,819,302,93,5.00,10,205,23,1526


Observations initiales : 
--- 
- 2505 entrées, 10 colonnes
- Types à modifier (trop de colonnes object)
- Valeurs manquantes sur :
    - Âge (143)
    - Prix_Billet (94)
    - Dépenses_Restaurant (102)
- Valeurs enregistrées sous différents formats (ex : STD, std, Standard, ...)

--- 
# 3. Nettoyage des données
---

## 3.1 Standardisation des textes

In [5]:
# Colonne "Visiteurs" : majuscule au début de chaque nom et prénom + suppression espace avant et après
df_clean["Visiteur"] = df_clean["Visiteur"].str.title().str.strip() 

# Colonne "Email" : tout en minuscule + suppression espace avant et après
df_clean["Email"] = df_clean["Email"].str.lower().str.strip() 

# Colonne "Pays" : majuscule au début de chaque nom + suppression espace avant et après
df_clean["Pays"] = df_clean["Pays"].str.title().str.strip() 

# Colonne "Type_Billets" : tout en majuscule + suppression espace avant et après + standardisation des items
df_clean["Type_Billet"] = df_clean["Type_Billet"].str.upper().str.strip().replace({"STD" : "STANDARD",
                                                                                   "ETUD" : "ETUDIANT", 
                                                                                   "FAM" : "FAMILLE",
                                                                                   "BILLET VIP" : "VIP",
                                                                                   "ÉTUDIANT" : "ETUDIANT",
                                                                                   "V I P" : "VIP"})

## 3.2 Standardisation des valeurs numériques

In [6]:
# Colonnes numériques à nettoyer
cols_num = ["Âge", "Prix_Billet", "Dépenses_Restaurant"]

# Remplacement des séparateurs décimaux et caractères parasites
for col in cols_num:
    # 1. Nettoyage des caractères parasites
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.replace('"', "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )

    # 2. Gestion des valeurs manquantes "encodées"
    df_clean[col] = df_clean[col].replace(["?", "nan", "None", ""], np.nan)

## 3.3 Conversion des types 

In [7]:
# Colonnes catégories 
for col in ["Pays", "Type_Billet", "Description", "Période"]:
   df_clean[col] = df_clean[col].astype("category")

# Colonnes float 
for col in ["Prix_Billet", "Dépenses_Restaurant"]:
   df_clean[col] = pd.to_numeric(df_clean[col]).round(2) # Arrondi au centime

# Colonne int
df_clean["Âge"] = pd.to_numeric(df_clean["Âge"])

# Colonne datetime
df_clean["Date_Visite"] = pd.to_datetime(df_clean["Date_Visite"])

---
# 4. Doublons
---

In [8]:
# Suppression des doublons selon le nom, l'adresse mail et la date de visite
df_clean = df_clean.drop_duplicates(subset = ["Visiteur", "Email", "Date_Visite"])

---
# 5. Outliers
---

### Prix_Billet

Recours à la méthode IQR pour détecter les valeurs extrêmes de la variable "Prix_Billet". Les valeurs sont d'abord regroupées par "Type_Billet" étant donné les variations de prix entre les différentes catégories de billets. Les effectifs des différents groupes ont été contrôlés préalablement et se sont révélés suffisants pour l'application de cette méthode.

In [9]:
col_group = "Type_Billet"
col_etudie = "Prix_Billet"

Q1 = df_clean.groupby(col_group, observed = True)[col_etudie].transform(lambda x: x.quantile(0.25))
Q3 = df_clean.groupby(col_group, observed = True)[col_etudie].transform(lambda x: x.quantile(0.75))
IQR = Q3 - Q1

val_aberrantes = df_clean[
    (df_clean[col_etudie] < Q1 - 1.5 * IQR) |
    (df_clean[col_etudie] > Q3 + 1.5 * IQR)
]

val_aberrantes.groupby(
    ["Type_Billet", "Période"],
    observed=True
)["Prix_Billet"].agg(
    ["count", "min", "median", "max"]
)

count     min   median     max
Type_Billet Période                                
ETUDIANT    Avant        1  114.51  114.510  114.51
FAMILLE     Après        5   62.12  155.890  627.06
            Avant        4   77.46  160.515  166.86
STANDARD    Après        8   27.72  112.025  261.67
            Avant        6   60.82   97.470  305.94
VIP         Après        9   30.21   51.770  489.27

Observations : 
---
- Grande dispersion sur la variable "Prix_Billet" avec plusieurs valeurs détectées comme extrêmes
- Mais "Prix_Billet" ne semble pas correspondre à un prix unitaire fixe mais à un montant total dépensé pour les billets
- Donc ce montant peut être influencée par le contexte d'achat (nombre de billets achetés, période)

Conclusion : 
---
- Ces outliers ne peuvent pas être interprétés catégoriquement comme des erreurs de saisie
- Elles semblent plutôt refléter des comportements d'achat différents que des anomalies
- Ces valeurs seront donc conservées pour la suite de l'analyse

### Dépenses_Restaurant

Détection des valeurs extrêmes de la variable "Dépenses_Restaurant" à l'aide de la méthode IQR. Les valeurs sont groupées par produit acheté (variable "Description") et par "Type_Billet". Cette dernière catégorie peut renseigner sur le comportement d'achat du client. De nouveau, la taille des groupes a été observée et jugée suffisante pour l'utilisation de cette méthode.

In [10]:
col_etudie = "Dépenses_Restaurant"
col_group = ["Description", "Type_Billet"]

Q1 = df_clean.groupby(col_group, observed = True)[col_etudie].transform(lambda x: x.quantile(0.25))
Q3 = df_clean.groupby(col_group, observed = True)[col_etudie].transform(lambda x: x.quantile(0.75))
IQR = Q3 - Q1

val_aberrantes = df_clean[
    (df_clean[col_etudie] < Q1 - 1.5 * IQR) |
    (df_clean[col_etudie] > Q3 + 1.5 * IQR)
]

print(f"Valeurs aberrantes détectées : {len(val_aberrantes)}")

Valeurs aberrantes détectées : 102


In [11]:
# Affichage des 10 plus grandes dépenses de restauration
val_aberrantes.sort_values(
    "Dépenses_Restaurant",
    ascending=False
).head(10)

,Visiteur,Email,Pays,Type_Billet,Âge,Prix_Billet,Dépenses_Restaurant,Description,Date_Visite,Période
1801,Fatima Haddad,fatima.haddad673@yahoo.fr,France,VIP,48.0,89.69,1048.04,Tacos,2025-05-31,Après
545,Kenza Silva,kenza.silva2376@gmail.com,France,STANDARD,22.0,47.18,167.70,Salade César,2025-06-16,Après
2194,Bob Petit,bob.petit524@gmail.com,Allemagne,STANDARD,30.0,45.18,141.53,Crêpe Sucre,2025-04-19,Après
2048,Nora Muller,nora.muller1507@yahoo.fr,Espagne,FAMILLE,19.0,130.03,99.71,Menu Enfant,2025-03-27,Après
581,Fatima Diaz,fatima.diaz184@outlook.com,Espagne,FAMILLE,59.0,92.01,97.11,Tacos,2025-01-07,Avant
2378,Kenza Gonzalez,kenza.gonzalez1183@gmail.com,France,VIP,27.0,86.31,86.66,Gaufre,2025-03-23,Après
2193,Jules Schmidt,jules.schmidt2362@outlook.com,Espagne,FAMILLE,32.0,148.09,85.73,Hot-Dog,2025-02-06,Avant
711,Fatima Gonzalez,fatima.gonzalez1590@gmail.com,France,FAMILLE,32.0,120.57,84.55,Glace Vanille,2025-05-18,Après
646,Bob Dupont,bob.dupont1399@yahoo.fr,France,FAMILLE,28.0,132.66,82.19,Menu Enfant,2025-01-27,Avant
2229,Zoe Bernard,zoe.bernard1243@yahoo.fr,France,FAMILLE,33.0,125.78,80.43,Smoothie Fraise,2025-02-03,Avant


Observations : 
--- 
- Il y a 102 valeurs aberrantes détectées via cette méthode pour cette variable
- De même que pour la variable "Prix_Billet", ces montants correspondent à des paniers globaux et non à des prix unitaires.
- Ces informations sont donc susceptibles d'être dépendantes des comportements d'achats (quantités achetées, famille nombreuses, groupe en voyage organisé, etc.)
- Sans plus d'informations, bien que ces valeurs soient atypiques, il n'est pas possible de conclure à une erreur de saisie
- Une valeur attire en revanche particulièrement l'attention : 1048,04 pour la ligne 1801

Conclusion : 
---
- Cette valeur est sans commune mesure d'une part avec les autres valeurs de la variable "Dépenses_Restaurant", et d'autre part, après observations du profil client associé, au prix payé pour les billet. Ce dernier montant est de 89,69, soit près de dix fois inférieur à la dépense de restauration. Ce ratio ne se retrouve pas dans les autres valeurs extrêmes. 
- Il semble pertinent de retirer cette valeur pour assurer la fiabilité de l'analyse.

In [12]:
# Suppression de la valeur jugée comme aberrante
df_clean = df_clean.drop(index = 1801)

---
# 6. Analyse et traitement des valeurs manquantes
---

### Âge

In [13]:
# Imputation par la médiane 
df_clean["Âge"] = (
    df_clean["Âge"]
    .fillna(df_clean["Âge"].median())
    .astype(int)
)

### Prix_Billet

In [14]:
# Imputation par la moyenne selon le type de billet et la période (Avant/Après la campagne)
df_clean["Prix_Billet"] = df_clean.groupby(
    ["Type_Billet", "Période"],
    observed=True
)["Prix_Billet"].transform(
    lambda x: x.fillna(x.mean())
)

### Dépenses_Restaurant

In [15]:
# Imputation par la moyenne en fonction de la description, c'est à dire le produit acheté, et la période
df_clean["Dépenses_Restaurant"] = df_clean.groupby(
    ["Description", "Période"],
    observed=True
)["Dépenses_Restaurant"].transform(
    lambda x: x.fillna(x.mean())
)

---
# 7. Contrôle qualité après nettoyage
---

In [16]:
print(f"Nombre de lignes supprimées : {len(df_raw) - len(df_clean)}")

Nombre de lignes supprimées : 6


In [17]:
df_clean.isna().sum()

Visiteur               0
Email                  0
Pays                   0
Type_Billet            0
Âge                    0
Prix_Billet            0
Dépenses_Restaurant    0
Description            0
Date_Visite            0
Période                0
dtype: int64

L'imputation a fonctionné

In [18]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2499 entries, 0 to 2504
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Visiteur             2499 non-null   object        
 1   Email                2499 non-null   object        
 2   Pays                 2499 non-null   category      
 3   Type_Billet          2499 non-null   category      
 4   Âge                  2499 non-null   int64         
 5   Prix_Billet          2499 non-null   float64       
 6   Dépenses_Restaurant  2499 non-null   float64       
 7   Description          2499 non-null   category      
 8   Date_Visite          2499 non-null   datetime64[ns]
 9   Période              2499 non-null   category      
dtypes: category(4), datetime64[ns](1), float64(2), int64(1), object(2)
memory usage: 147.6+ KB


Toutes les variables ont été converties vers le type approprié.

In [19]:
df_clean["Type_Billet"].value_counts()

Type_Billet
STANDARD    1152
VIP          534
ETUDIANT     494
FAMILLE      319
Name: count, dtype: int64

La standardisation des types de billet a réussi

Indicateurs de qualité des données

| Contrôle | Avant | Après |
|-----------|--------|--------|
| Nombre de lignes | 2505 | 2499 |
| Valeurs manquantes Âge | 143 | 0 |
| Valeurs manquantes Prix_Billet | 94 | 0 |
| Valeurs manquantes Dépenses_Restaurant | 102 | 0 |
| Doublons détectés | 5 | 0 |
| Valeurs aberrantes supprimées | 1 | 0 |

---
# 8. Export des données
---

In [20]:
df_clean.to_excel("../data/cleaned/funpark_data_cleaned.xlsx")